# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-08-10T03:59:05.233Z

## 1. Install dependencies

In [ ]:
%pip install autora-core==5.0.3 autora-theorist-darts==1.1.0 autora-synthetic==2.2.0

## 2. Imports

In [ ]:
from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection, IV, DV
from autora.experimentalist.random import pool as random_pooler, sample as random_sampler
from autora.experiment_runner.synthetic.abstract.lmm import lmm_experiment
from autora.theorist.darts.regressor import DARTSRegressor

import pandas as pd
import numpy as np

## 3. Component definitions

In [ ]:
# Random Pooler
@on_state()
def random_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=random_pooler(variables, num_samples=5, replace=True))

In [ ]:
# Random Sampler
@on_state()
def random_sampler_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=random_sampler(conditions=conditions, num_samples=num_samples, replace=False))

In [ ]:
# Linear Mixed Model Experiment (Synthetic, Abstract)
@on_state()
def linear_mixed_model_experiment_on_state(conditions: pd.DataFrame) -> Delta:
    runner = lmm_experiment(
        formula="rt ~ 1 + x1", fixed_effects={'Intercept': 0., 'x1': 2.},
        # TODO: adjust the variable names and ranges below for your experiment
        X=[
            IV(name="x1", allowed_values=np.linspace(-10, 10, 100), value_range=(-10, 10)),
        ])
    assert runner.run is not None
    return Delta(experiment_data=runner.run(conditions=conditions))

In [ ]:
# DARTS Regressor
darts_regressor_on_state = estimator_on_state(DARTSRegressor(batch_size=64, num_graph_nodes=2, output_type="real", classifier_weight_decay=0.01, darts_type="original", param_updates_per_epoch=10, param_updates_for_sampled_model=100, param_learning_rate_max=0.025, param_learning_rate_min=0.01, param_momentum=0.9, arch_updates_per_epoch=1, arch_learning_rate_max=0.003, arch_weight_decay=0.0001, arch_weight_decay_df=0.0003, arch_weight_decay_base=0, arch_momentum=0.9, fair_darts_loss_weight=1, max_epochs=10, grad_clip=5, primitives=["none", "add", "subtract", "linear", "linear_logistic", "linear_relu"], train_classifier_coefficients=False, train_classifier_bias=False, sampling_strategy="max"))

## 4. Run the workflow

In [ ]:
# Variables are created and governed by the experiment runner
runner = lmm_experiment(
    formula="rt ~ 1 + x1", fixed_effects={'Intercept': 0., 'x1': 2.},
    # TODO: adjust the variable names and ranges below for your experiment
    X=[
        IV(name="x1", allowed_values=np.linspace(-10, 10, 100), value_range=(-10, 10)),
    ])
assert runner.variables is not None
variables = runner.variables

# Initialize state
state = StandardState(variables=variables)

# Main experiment loop (1 cycles)
for i in range(1):
    print(f'Cycle {i}')

    # Random Pooler
    state = random_pooler_on_state(state)

    # Random Sampler
    state = random_sampler_on_state(state, num_samples=1)

    # Linear Mixed Model Experiment (Synthetic, Abstract)
    state = linear_mixed_model_experiment_on_state(state)

    # DARTS Regressor
    state = darts_regressor_on_state(state)


print("Workflow completed!")
state